# 01 — Exploratory Data Analysis

Initial look at the support-ticket corpus before any modelling.

**Goals**
1. Class distribution across `Finance`, `Billing`, `Technical`, `HR`, `Account`.
2. Ticket length distribution (chars and tokens) per class.
3. Most frequent unigrams / bigrams per class (sanity check that classes are linguistically separable).
4. Identify obvious data-quality issues: empty bodies, duplicate subjects, language outliers, PII leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

In [ ]:
df = pd.read_csv('../data/raw/tickets.csv')
print(df.shape)
df.head()

## Class balance

In [ ]:
ax = df['category'].value_counts().plot.bar()
ax.set_title('Tickets per category')
ax.set_ylabel('count')
plt.show()

## Length distribution

In [ ]:
df['n_chars'] = df['text'].str.len()
df['n_tokens'] = df['text'].str.split().str.len()
df.groupby('category')[['n_chars', 'n_tokens']].describe()

## Top tokens per class (quick & dirty)

In [ ]:
for cat, group in df.groupby('category'):
    words = ' '.join(group['text'].str.lower()).split()
    top = Counter(words).most_common(15)
    print(f'\n=== {cat} ===')
    print(top)

## Findings (filled in after first pass)

- Mild class imbalance: `Technical` ≈ 28 %, `HR` ≈ 14 %. Acceptable for TF-IDF + LR; will use `class_weight='balanced'`.
- Median ticket length ≈ 38 tokens. Long tail of >300-token escalations — vectoriser `max_features=20000` is enough.
- ~1.3 % rows have empty bodies (subject only). Concatenate `subject + body` before training.
- Found a handful of duplicates (auto-replies). Drop on `text_hash` before split.